In [ ]:
%pip install -q "transformers>=4.46,<5" sentence-transformers rank_bm25 sentencepiece protobuf scikit-learn pandas pyarrow joblib youtube-transcript-api "gradio>=5,<6"

In [ ]:
import os, re, json, html as H, collections, joblib, numpy as np, pandas as pd, torch
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, AutoModelForSequenceClassification
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = "/content/drive/MyDrive/veritas"
except ImportError:
    DRIVE = "veritas_local"
DEV = "cuda" if torch.cuda.is_available() else "cpu"
LABELS = ["SUPPORTS", "REFUTES", "NEI"]
GATE = 0.15
MAX_CLAIMS = 30

cfg = json.load(open(f"{DRIVE}/retrieval/retrieval_config.json"))
EMB_MODEL, QUERY_PREFIX, TOPK, NCAND, RERANK_MODEL = cfg["emb_model"], cfg["query_prefix"], cfg["topk"], cfg["ncand"], cfg["rerank_model"]
corpus = pd.read_parquet(f"{DRIVE}/retrieval/corpus.parquet"); D = np.load(f"{DRIVE}/retrieval/corpus_emb.npy")
agg = joblib.load(f"{DRIVE}/retrieval/aggregator.joblib")
assert len(corpus) == len(D), "corpus and embeddings are out of sync"

half = lambda m: m.half() if DEV == "cuda" else m
vtok = AutoTokenizer.from_pretrained(f"{DRIVE}/verdict_model_v2")
vmodel = half(AutoModelForSequenceClassification.from_pretrained(f"{DRIVE}/verdict_model_v2").to(DEV).eval())
ctok = AutoTokenizer.from_pretrained(f"{DRIVE}/claim_detector_v2")
cmodel = half(AutoModelForSequenceClassification.from_pretrained(f"{DRIVE}/claim_detector_v2").to(DEV).eval())
CLAIM_THR = json.load(open(f"{DRIVE}/claim_detector_v2/extract_config.json"))["threshold"]
st = SentenceTransformer(EMB_MODEL, device=DEV); st.max_seq_length = 256
rr = CrossEncoder(RERANK_MODEL, max_length=384, device=DEV)
if DEV == "cuda": rr.model.half()

tok_re = re.compile(r"[a-z0-9]+")
STOP = set("a an and are as at be by for from has have in is it its of on or that the this to was were will with".split())
toks = lambda t: [w for w in tok_re.findall(t.lower()) if w not in STOP]
bm25 = BM25Okapi([toks(t) for t in corpus.search_text])
print("loaded | device:", DEV, "| pool:", len(corpus), "| claim threshold:", CLAIM_THR, "| gate:", GATE)

In [ ]:
def rank_bm25_q(qs, n=100): return [np.argsort(-bm25.get_scores(toks(q)))[:n] for q in qs]
def rank_dense_q(qs, n=100):
    Q = st.encode([QUERY_PREFIX + q for q in qs], batch_size=64, normalize_embeddings=True, convert_to_numpy=True)
    return [np.argsort(-s)[:n] for s in Q @ D.T]
def fuse(r1, r2, n=100, k=60):
    out = []
    for a, b in zip(r1, r2):
        sc = collections.defaultdict(float)
        for r in (a, b):
            for rank, i in enumerate(r): sc[i] += 1.0 / (k + rank + 1)
        out.append(np.array(sorted(sc, key=sc.get, reverse=True)[:n]))
    return out

def rerank(texts, rankings, ncand=NCAND, bs=64):
    pairs = [(t, corpus.search_text.iat[di]) for t, r in zip(texts, rankings) for di in r[:ncand]]
    sc = np.asarray(rr.predict(pairs, batch_size=bs, show_progress_bar=False), dtype="float64").reshape(len(texts), -1)
    if sc.min() < 0 or sc.max() > 1: sc = 1 / (1 + np.exp(-sc))
    nr, ns = [], []
    for r, s in zip(rankings, sc):
        o = np.argsort(-s); nr.append(r[:ncand][o]); ns.append(s[o])
    return nr, np.array(ns)

@torch.no_grad()
def verdict_probs(pairs, bs=32, max_len=384):
    out = []
    for i in range(0, len(pairs), bs):
        b = pairs[i:i + bs]
        enc = vtok([c for c, _ in b], [d for _, d in b], truncation="only_second", max_length=max_len, padding=True, return_tensors="pt").to(DEV)
        out.append(torch.softmax(vmodel(**enc).logits.float(), -1).cpu().numpy())
    return np.concatenate(out)

def make_feats(P, S):
    S = S[:, :P.shape[1]]; w = S / (S.sum(1, keepdims=True) + 1e-9); wp = (w[:, :, None] * P).sum(1)
    return np.column_stack([P[:, :, 0].max(1), P[:, :, 1].max(1), P[:, :, 0].mean(1), P[:, :, 1].mean(1),
                            P[:, 0, 0], P[:, 0, 1], P[:, 0, 2], P[:, :, 2].mean(1), wp, S[:, 0], S[:, 0] - S[:, 1], S.mean(1)])

def check_claims(texts, show=3):
    rk, S = rerank(texts, fuse(rank_bm25_q(texts), rank_dense_q(texts)))
    pairs = [(t, corpus.text.iat[di]) for t, r in zip(texts, rk) for di in r[:TOPK]]
    P = verdict_probs(pairs).reshape(len(texts), TOPK, 3)
    A = agg.predict_proba(make_feats(P, S)); order = list(agg.classes_); out = []
    for i, t in enumerate(texts):
        probs = {LABELS[k]: float(A[i][order.index(k)]) if k in order else 0.0 for k in range(3)}
        gated = bool(S[i, 0] < GATE)
        verdict = "NEI" if gated else max(probs, key=probs.get)
        ev = [dict(title=corpus.title.iat[di] or "(untitled)", rerank=float(S[i, j]), passage=LABELS[int(P[i, j].argmax())],
                   p=float(P[i, j].max()), excerpt=corpus.text.iat[di][:420]) for j, di in enumerate(rk[i][:show])]
        out.append(dict(claim=t, verdict=verdict, confidence=None if gated else probs[verdict], gated=gated, probs=probs, evidence=ev))
    return out

@torch.no_grad()
def claim_probs(sents, bs=64):
    out = []
    for i in range(0, len(sents), bs):
        b = ctok(sents[i:i + bs], truncation=True, max_length=128, padding=True, return_tensors="pt").to(DEV)
        out.append(torch.softmax(cmodel(**b).logits.float(), -1)[:, 1].cpu().numpy())
    return np.concatenate(out) if out else np.array([])

def split_transcript(text, max_words=40, window=25):
    text = re.sub(r"\[?\(?\b\d{1,2}:\d{2}(?::\d{2})?\b\]?\)?", " ", text)
    text = re.sub(r"\s+", " ", text).strip(); out = []
    for s in re.split(r"(?<=[.!?])\s+", text):
        w = s.split()
        if len(w) <= max_words:
            if len(w) >= 4: out.append(s)
        else:
            out += [" ".join(w[i:i + window]) for i in range(0, len(w), window) if len(w[i:i + window]) >= 4]
    return out

In [ ]:
V = {"SUPPORTS": ("Supported by evidence", "#10b981", "#ecfdf5"), "REFUTES": ("Contradicted by evidence", "#ef4444", "#fef2f2"),
     "NEI": ("Not enough evidence", "#6b7280", "#f3f4f6")}
conf_label = lambda c: "high" if c >= 0.7 else "medium" if c >= 0.5 else "low"
INK = "color:#111827 !important;"
WRAP = lambda h: "<div style='background:#ffffff;padding:12px;border-radius:8px;color:#111827 !important'>" + h + "</div>"

def card(r, i):
    name, border, bg = V[r["verdict"]]
    if r["gated"]:   conf = "no relevant abstract found in the evidence pool"
    else:            conf = f"confidence {conf_label(r['confidence'])} ({r['confidence']:.0%})"
    ev = "".join(
        f"<details style='margin:4px 0'><summary style='cursor:pointer;{INK}font-size:13px'>"
        f"<b style='{INK}'>{H.escape(e['title'][:110])}</b> &nbsp;<span style='color:#4b5563 !important'>reads as {e['passage'].lower()} ({e['p']:.0%}) · relevance {e['rerank']:.2f}</span></summary>"
        f"<div style='font-size:12px;color:#374151 !important;padding:4px 0 4px 12px'>{H.escape(e['excerpt'])}…</div></details>" for e in r["evidence"])
    return (f"<div style='border-left:6px solid {border};background:{bg};padding:10px 14px;margin:10px 0;border-radius:6px;{INK}'>"
            f"<div style='font-size:12px;color:#4b5563 !important'>CLAIM {i}</div>"
            f"<div style='font-size:15px;margin:2px 0 6px 0;{INK}'>{H.escape(r['claim'])}</div>"
            f"<div style='font-size:14px;{INK}'><b style='color:{border} !important'>{name}</b> &nbsp;·&nbsp; {conf}</div>"
            f"<div style='margin-top:6px;{INK}'>{ev}</div></div>")

def analyze(transcript, thr=CLAIM_THR):
    sents = split_transcript(transcript or "")
    if not sents: return WRAP("<i>Paste a transcript first.</i>"), ""
    p = claim_probs(sents); flagged = [i for i, x in enumerate(p) if x >= thr]
    note = ""
    if len(flagged) > MAX_CLAIMS:
        flagged = sorted(sorted(flagged, key=lambda i: -p[i])[:MAX_CLAIMS]); note = f" (showing the {MAX_CLAIMS} most claim-like)"
    res = check_claims([sents[i] for i in flagged]) if flagged else []
    by = {i: r for i, r in zip(flagged, res)}
    cnt = collections.Counter(r["verdict"] for r in res)
    summary = (f"<div style='{INK}font-size:14px;margin-bottom:6px'><b>{len(sents)}</b> sentences · <b>{len(flagged)}</b> checkable claims{note} · "
               f"<span style='color:#10b981 !important'><b>{cnt['SUPPORTS']}</b> supported</span> · <span style='color:#ef4444 !important'><b>{cnt['REFUTES']}</b> contradicted</span> · "
               f"<span style='color:#6b7280 !important'><b>{cnt['NEI']}</b> not enough evidence</span></div>")
    cards = "".join(card(r, n + 1) for n, r in enumerate(res)) or "<i>No checkable claims found. Try lowering the claim threshold.</i>"
    spans = []
    for i, s in enumerate(sents):
        if i in by:
            _, border, bg = V[by[i]["verdict"]]
            spans.append(f"<span style='background:{bg};border-bottom:3px solid {border};padding:1px 3px;{INK}'>{H.escape(s)}</span>")
        else:
            spans.append(f"<span style='color:#6b7280 !important'>{H.escape(s)}</span>")
    view = "<div style='line-height:1.9;font-size:14px'>" + " ".join(spans) + "</div>"
    return WRAP(summary + cards), WRAP(view)

In [ ]:
def fetch_captions(url):
    m = re.search(r"(?:v=|youtu\.be/|shorts/|embed/)([A-Za-z0-9_-]{11})", url or "")
    if not m: return "", "Could not find a video id in that link."
    vid = m.group(1)
    try:
        from youtube_transcript_api import YouTubeTranscriptApi
        try:    segs = [s.text for s in YouTubeTranscriptApi().fetch(vid)]
        except AttributeError: segs = [s["text"] for s in YouTubeTranscriptApi.get_transcript(vid)]
        return " ".join(segs), f"Fetched {len(segs)} caption lines."
    except Exception as e:
        return "", f"Could not fetch captions ({type(e).__name__}). Paste the transcript instead."

In [ ]:
DEMO1 = '''Hey everyone, welcome back to the channel. Today I want to talk about sleep and supplements. So first, I take magnesium glycinate every night and honestly it cures my insomnia. Studies show that vitamin D supplementation reduces the risk of respiratory infections. I personally think cold plunges are amazing. Masks do not reduce the spread of COVID-19, that's just a fact. Intermittent fasting is the best diet in my opinion. Drinking lemon water every morning cures cancer. Make sure you like and subscribe. The Eiffel Tower is 330 meters tall. Statins cause more harm than good for most people. Anyway, let me know what you think in the comments.'''
DEMO2 = '''Okay so, um, last month my doctor told me my cholesterol was high. A lot of people say eggs raise your cholesterol but the research really doesn't back that up. I swapped sugar for honey and my energy is way better. Drinking eight glasses of water a day is necessary for good health. Seed oils are poison, they cause inflammation throughout the body. Honestly this smoothie recipe is so good. Sunscreen causes more cancer than it prevents. If you walk ten thousand steps a day you will live longer. Alright, that's it for today, thanks for watching.'''
for name, T in [("DEMO1", DEMO1), ("DEMO2", DEMO2)]:
    sents = split_transcript(T); p = claim_probs(sents); fl = [i for i, x in enumerate(p) if x >= CLAIM_THR]
    print(f"\n{name}: {len(sents)} sentences, {len(fl)} claims")
    for i, r in zip(fl, check_claims([sents[i] for i in fl])):
        conf = "no relevant abstract" if r["gated"] else f"{r['confidence']:.2f}"
        print(f"  {r['verdict']:8s} {conf:>20s} | {r['claim'][:80]}")

In [ ]:
import gradio as gr

LIMITS = '''**How it works.** A claim detector picks checkable sentences. Each claim is searched against a pool of **5,505 research abstracts** (SciFact + HealthVer), re-ranked, and judged by a fine-tuned DeBERTa verdict model.

**Honest limits**
- **Coverage:** the evidence pool is small. Claims on topics outside it return *not enough evidence*, which means "not found here", not "false".
- **Accuracy:** on held-out HealthVer claims the system scores about 0.6 macro-F1 (0.705 if handed the right abstract). Expect mistakes. Confidence labels are rough guides.
- **Claim detection** was trained on political debates, health claims and a small amount of synthetic text. It over-flags personal anecdotes.
- Research abstracts are not medical advice. Check with a clinician.'''

with gr.Blocks(title="Veritas: health claim checker") as demo:
    gr.Markdown("# Veritas — health claim checker\nPaste a video transcript. Veritas finds the health claims and checks them against research abstracts.")
    with gr.Row():
        with gr.Column(scale=1):
            tx = gr.Textbox(label="Transcript", lines=12, placeholder="Paste a transcript here…")
            with gr.Row():
                url = gr.Textbox(label="…or a YouTube link", scale=3); fetch = gr.Button("Fetch captions", scale=1)
            status = gr.Markdown("")
            thr = gr.Slider(0.3, 0.99, value=CLAIM_THR, step=0.01, label="Claim threshold (lower = flag more sentences)")
            go = gr.Button("Check claims", variant="primary")
            gr.Examples(examples=[[DEMO1], [DEMO2]], inputs=[tx], label="Invented example transcripts")
        with gr.Column(scale=1):
            out = gr.HTML(label="Results")
            with gr.Accordion("Transcript view", open=False): view = gr.HTML()
    with gr.Accordion("How it works and honest limits", open=False): gr.Markdown(LIMITS)
    fetch.click(fetch_captions, [url], [tx, status])
    go.click(analyze, [tx, thr], [out, view])

demo.launch(share=True, debug=False)